# Lifestyle Factors and Diabetes Risk: A Bayesian Analysis

This notebook investigates associations between lifestyle behaviours, including physical activity, sedentary time, and sleep duration, and high diabetes risk in a Mexican population sample from ENSANUT 2024. We fit both a frequentist logistic regression and a Bayesian logistic regression using PyMC, and use posterior samples to simulate the effect of population-level interventions.

In [ ]:
!pip install git+https://github.com/BirkhoffG/causalgraphicalmodels.git

In [ ]:
%pylab inline
%config InlineBackend.figure_format = 'retina'
from ipywidgets import interact
import scipy.stats as stats
import pandas as pd
import pymc as pm
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.gridspec as gridspec

import arviz as az ## This is new, but it is installed along with PyMC

## This is new for working with DAGs, you will have to install it
## uncomment the next box and run it
import causalgraphicalmodels as cgm
from causalgraphicalmodels import CausalGraphicalModel
import graphviz

from scipy.special import expit
import matplotlib.ticker as mtick

import warnings
warnings.simplefilter(action="ignore", category=FutureWarning)

def credible_interval_from_samples(samples, prob):
    """`samples` can be an nd array. Assume that all of the dimensions
    except for the last index parameters while the last (rightmost)
    dimension indexes the samples."""
    samples_sorted = sort(samples, axis=-1)
    N_samples = samples.shape[-1]
    index = int(N_samples*(1 - prob)/2)
    lower = samples_sorted[..., index]
    upper = samples_sorted[..., -index]
    ## quantile(x, [(1 - prob)/2, (1 + prob)/2], axis=-1)
    return lower, upper

In [ ]:
main_path = "https://raw.githubusercontent.com/hhhaiying/diabetes-lifestyle-analysis/refs/heads/main/data/Diabetes_Mexico.csv"
sub_path = "https://raw.githubusercontent.com/hhhaiying/diabetes-lifestyle-analysis/refs/heads/main/data/actividad_fisica_ensanut2024.csv"

## 2. Data Loading & Preparation

We pulling two raw ENSANUT 2024 files from GitHub and merging them on a shared participant ID (`folio_int`): the main diabetes module and the physical activity sub-module. From there, we clean up the lifestyle variables, replacing survey non-response codes with `NaN`, converting sleep duration into an ordered category, and combining the sedentary hours and minutes columns into a single daily total. Where BMI is missing but weight and height are both available, we fill it in directly.

The final analytic sample keeps only participants with complete data on all lifestyle exposures and core covariates. We then create a few modelling-ready variables: a binary flag for any vigorous activity, age and sedentary hours centred at their means, and sleep-duration dummies with 7 hours as the reference category.

In [ ]:
cols_interest = ["fa0400", "fa0401", "fa0403", "fa0405", "fa0407h", "fa0407m"]

def find_id_col(cols):
    for c in ["folio_int", "FOLIO_INT"]:
        if c in cols:
            return c
    lower_map = {c.lower(): c for c in cols}
    if "folio_int" in lower_map:
        return lower_map["folio_int"]
    raise ValueError("Could not find folio_int/FOLIO_INT in columns.")

def clean_id(s: pd.Series) -> pd.Series:
    out = s.astype(str).str.strip()
    out = out.str.replace(r"\.0$", "", regex=True)
    out = out.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return out

main_cols = pd.read_csv(main_path, nrows=0).columns
sub_cols  = pd.read_csv(sub_path,  nrows=0).columns
id_main = find_id_col(main_cols)
id_sub  = find_id_col(sub_cols)

main = pd.read_csv(main_path, low_memory=False)
sub  = pd.read_csv(sub_path, usecols=[id_sub] + cols_interest, low_memory=False)

main["_id"] = clean_id(main[id_main])
sub["_id"]  = clean_id(sub[id_sub])

main = main[main["_id"].notna()].copy()
sub  = sub[sub["_id"].notna()].copy()

dup = sub["_id"].duplicated(keep=False)
if dup.any():
    print("WARNING: subdataset has duplicated _id rows:", int(dup.sum()))
    sub = sub.sort_values("_id").drop_duplicates("_id", keep="first")

joined = main.merge(
    sub[["_id"] + cols_interest],
    on="_id",
    how="left",
    validate="m:1"
)

df = joined.drop(columns=["_id"])

print("joined shape:", df.shape)
print("matched rows:", int(joined[cols_interest].notna().any(axis=1).sum()))

In [ ]:
pa_vars = ["fa0401", "fa0403", "fa0405"]

# Age filter
df["edad"] = pd.to_numeric(df["edad"], errors="coerce")
df = df[df["edad"].between(0, 120)].copy()

for c in cols_interest:
    if c not in df.columns:
        raise ValueError(f"Missing column: {c}")

# Keep raw copies
for c in cols_interest:
    df[c + "_raw"] = df[c]

# Convert to numeric
for c in cols_interest:
    df[c] = pd.to_numeric(df[c], errors="coerce")

missing_codes = [88, 99]
unable_code = 55

# Sleep
df["sleep_cat"] = df["fa0400"].replace(missing_codes, np.nan)

sleep_map = {1: "<=5h", 2: "6h", 3: "7h", 4: "8h", 5: ">=9h"}
df["sleep_label"] = df["sleep_cat"].map(sleep_map)

# Physical activity
for c in pa_vars:
    df[c + "_unable_flag"] = (df[c] == unable_code).astype(int)
    df[c + "_clean"] = df[c].replace(missing_codes + [unable_code], np.nan)

df["vig_days"]  = df["fa0401_clean"]
df["mod_days"]  = df["fa0403_clean"]
df["walk_days"] = df["fa0405_clean"]

# Sedentary time
df["fa0407h_clean"] = df["fa0407h"].replace(missing_codes, np.nan)
df["fa0407m_clean"] = df["fa0407m"].replace(missing_codes, np.nan)

df.loc[
    df["fa0407h_clean"].notna() & df["fa0407m_clean"].isna(),
    "fa0407m_clean"
] = 0

df.loc[
    df["fa0407m_clean"].notna() & ~df["fa0407m_clean"].between(0, 59),
    "fa0407m_clean"
] = np.nan

df["sedentary_min"] = df["fa0407h_clean"] * 60 + df["fa0407m_clean"]

# Complete-case flag
model_vars = ["sleep_cat", "vig_days", "mod_days", "walk_days", "sedentary_min"]
df["lifestyle_complete_after_recode"] = df[model_vars].notna().all(axis=1)

print("\n=== COMPLETE AFTER RECODE ===")
print(f"n={int(df['lifestyle_complete_after_recode'].sum())}, rate={df['lifestyle_complete_after_recode'].mean():.3f}")

In [ ]:
required_cols = [
    "folio_i", "folio_int", "riesgo_diabetes_cat", "edad", "sexo", "Ciudad",
    "sleep_cat", "vig_days", "mod_days", "walk_days", "sedentary_min",
    "lifestyle_complete_after_recode", "Peso", "Estatura", "imc"
]

for c in required_cols:
    if c not in df.columns:
        raise ValueError(f"Missing column: {c}")

analysis_df = df[df["lifestyle_complete_after_recode"] == True].copy()

# Clean sex
analysis_df["sexo_raw"] = analysis_df["sexo"]
analysis_df["sexo"] = analysis_df["sexo"].astype(str).str.strip()
sex_map = {"Hombre": 1, "Mujer": 2}
analysis_df["sexo"] = analysis_df["sexo"].replace(sex_map)
analysis_df["sexo"] = pd.to_numeric(analysis_df["sexo"], errors="coerce")

# Clean age
analysis_df["edad_raw"] = analysis_df["edad"]
analysis_df["edad"] = pd.to_numeric(analysis_df["edad"], errors="coerce")
analysis_df.loc[~analysis_df["edad"].between(0, 120), "edad"] = pd.NA

# Convert numeric columns
other_num_cols = [
    "riesgo_diabetes_cat", "sleep_cat", "vig_days", "mod_days",
    "walk_days", "sedentary_min", "Peso", "Estatura", "imc"
]
for c in other_num_cols:
    analysis_df[c] = pd.to_numeric(analysis_df[c], errors="coerce")

# Plausibility checks
analysis_df.loc[~analysis_df["Peso"].between(20, 300), "Peso"] = pd.NA
analysis_df.loc[~analysis_df["Estatura"].between(100, 250), "Estatura"] = pd.NA
analysis_df.loc[~analysis_df["imc"].between(10, 80), "imc"] = pd.NA

# Fill missing BMI
height_m = analysis_df["Estatura"] / 100
bmi_from_hw = analysis_df["Peso"] / (height_m ** 2)
can_fill_bmi = (
    analysis_df["imc"].isna() &
    analysis_df["Peso"].notna() &
    analysis_df["Estatura"].notna() &
    height_m.notna() &
    (height_m > 0)
)
analysis_df.loc[can_fill_bmi, "imc"] = bmi_from_hw[can_fill_bmi]
analysis_df.loc[~analysis_df["imc"].between(10, 80), "imc"] = pd.NA

print("BMI missing after fill:", analysis_df["imc"].isna().sum())

# Drop missing core covariates
analysis_df = analysis_df.dropna(
    subset=["riesgo_diabetes_cat", "edad", "sexo", "Ciudad"]
).copy()

# Binary outcome
analysis_df["high_risk"] = (analysis_df["riesgo_diabetes_cat"] == 2).astype(int)

print("\n=== MAIN ANALYTIC SAMPLE ===")
print("n:", len(analysis_df))

print("\n=== MISSING CHECK IN ANALYTIC SAMPLE ===")
print(analysis_df[["riesgo_diabetes_cat", "edad", "sexo", "Ciudad", "Peso", "Estatura", "imc"]].isna().sum())

print("\n=== BINARY OUTCOME DISTRIBUTION ===")
outcome_counts = analysis_df["high_risk"].value_counts(dropna=False).sort_index()
outcome_props  = analysis_df["high_risk"].value_counts(normalize=True, dropna=False).sort_index()
print(pd.DataFrame({"count": outcome_counts, "prop": outcome_props}))

# Rename to English
analysis_df = analysis_df.rename(columns={
    "riesgo_diabetes_cat": "diabetes_risk_cat",
    "edad": "age",
    "sexo": "sex",
    "Ciudad": "city",
    "Peso": "weight_kg",
    "Estatura": "height_cm",
    "imc": "bmi"
})

In [ ]:
required = ["high_risk", "age", "sex", "sleep_cat","vig_days", "mod_days", "walk_days", "sedentary_min"]
model_df = analysis_df[required].copy()
for c in required:
    model_df[c] = pd.to_numeric(model_df[c], errors="coerce")
model_df = model_df.dropna().copy()

# recoding
model_df["female"] = (model_df["sex"] == 2).astype(int)
model_df["age_c"] = model_df["age"] - model_df["age"].mean()
model_df["sedentary_hours"] = model_df["sedentary_min"] / 60
model_df["sedentary_hours_c"] = model_df["sedentary_hours"] - model_df["sedentary_hours"].mean()

## vig_days: 71.2% zeros -> binary (any vigorous activity vs none)
model_df["vig_active"] = (model_df["vig_days"] > 0).astype(int)

## sleep dummies, reference = 7h
sleep_label_map = {1: "<=5h", 2: "6h", 3: "7h", 4: "8h", 5: ">=9h"}
sleep_order = ["<=5h", "6h", "7h", "8h", ">=9h"]
model_df["sleep_label"] = model_df["sleep_cat"].map(sleep_label_map)
sleep_dummies = pd.get_dummies(model_df["sleep_label"], prefix="sleep", drop_first=False)
sleep_dummies = sleep_dummies.drop(columns=["sleep_7h"]).astype(int)

print(f"Sample size : {len(model_df)}")
print(f"High risk : {model_df['high_risk'].sum()} ({model_df['high_risk'].mean():.1%})")
print(f"Low risk : {(model_df['high_risk']==0).sum()}")
model_df.head()

## 3. Causal Structure (DAG)

We specify our causal assumptions as a directed acyclic graph (DAG). Age and sex are treated as baseline confounders that influence both the lifestyle exposures (physical activity, sedentary time, sleep) and diabetes risk directly. Physical activity also affects sedentary time. An unobserved node `U_Diet` is included to acknowledge unmeasured confounding。

In [ ]:
dot = graphviz.Digraph()
dot.attr(nodesep='0.8', ranksep='1.0')

with dot.subgraph() as top:
    top.attr(rank='same')
    top.node('Sex')
    top.node('Age')
    top.node('U_Diet', style='dashed')

with dot.subgraph() as middle:
    middle.attr(rank='same')
    middle.node('Sleep Duration')
    middle.node('Physical Activity')
    middle.node('Sedentary Time')

with dot.subgraph() as bottom:
    bottom.attr(rank='same')
    bottom.node('Diabetes Risk')

# vertical edges
edges = [
    ('Age', 'Sedentary Time'), ('Age', 'Physical Activity'), ('Age', 'Sleep Duration'),
    ('Sex', 'Sedentary Time'), ('Sex', 'Physical Activity'), ('Sex', 'Sleep Duration'),
    ('Age', 'Diabetes Risk'), ('Sex', 'Diabetes Risk'),
    ('Sedentary Time', 'Diabetes Risk'), ('Physical Activity', 'Diabetes Risk'), ('Sleep Duration', 'Diabetes Risk')
]

for edge in edges:
    dot.edge(edge[0], edge[1])

# horizontal edges
dot.edge('Sleep Duration', 'Physical Activity', style='invis')
dot.edge('Physical Activity', 'Sleep Duration', constraint='false')
dot.edge('Physical Activity', 'Sedentary Time')

# unobserved edges with dashed line
dot.edge('U_Diet', 'Sedentary Time', style='dashed')
dot.edge('U_Diet', 'Diabetes Risk', style='dashed')

dot

## 4. Exploratory Data Analysis

Before modelling, we check for zero-inflation in the activity variables, assess inter-variable correlations using Spearman's ρ, and compute variance inflation factors to screen for multicollinearity. We also examine the distribution of each lifestyle variable and its unadjusted association with the binary outcome.


In [ ]:
activity_vars = ["vig_days", "mod_days", "walk_days"]
act_labels = {
    "vig_days":  "Vigorous activity days/week",
    "mod_days":  "Moderate activity days/week",
    "walk_days": "Walking days/week",
}

fig, axes = subplots(1, 3, figsize=(13, 4))
fig.suptitle("Activity variable distributions", fontsize=24)

for ax, var in zip(axes, activity_vars):
    counts = model_df[var].value_counts().sort_index()
    pct_zero = (model_df[var] == 0).mean() * 100

    ax.bar(counts.index, counts.values)
    ax.set_title(act_labels[var], fontsize=10)
    ax.set_xlabel("Days per week", fontsize=9)
    ax.set_ylabel("Count", fontsize=9)
    ax.set_xticks(range(0, 8))
    ax.text(0.5, 0.95, f"0-days: {pct_zero:.1f}%",
        transform=ax.transAxes, ha="center", va="top",
        fontsize=9,
        bbox=dict(facecolor="white", alpha=0.7))

tight_layout()

In [ ]:
print("=== ZERO-INFLATION SUMMARY ===")
for var in activity_vars:
    print(f"  {var:12s}: {(model_df[var]==0).mean()*100:.1f}% zeros  "
          f"median={model_df[var].median():.0f}  mean={model_df[var].mean():.1f}")

print("\n=== SPEARMAN CORRELATIONS ===")
pairs = [("vig_days","mod_days"), ("vig_days","walk_days"), ("mod_days","walk_days")]
for a, b in pairs:
    r, p = stats.spearmanr(model_df[a], model_df[b])
    print(f"  {a} vs {b}: r = {r:.3f}, p = {p:.4f}")

print("\n=== VIF ===")
model_df["age_c_tmp"]  = model_df["age"] - model_df["age"].mean()
model_df["sed_c_tmp"]  = model_df["sedentary_hours"] - model_df["sedentary_hours"].mean()
X_vif = sm.add_constant(model_df[["age_c_tmp", "female", "sleep_cat",
                                    "sed_c_tmp", "vig_days", "mod_days", "walk_days"]])
vif_data = pd.DataFrame({
    "variable": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i)
            for i in range(X_vif.shape[1])]
})
print(vif_data.to_string(index=False))

In [ ]:
fig2 = figure(figsize=(13, 8))
gs2  = gridspec.GridSpec(2, 3, hspace=0.45, wspace=0.35)
fig2.suptitle("Variable associations with high diabetes risk", fontsize=24)

overall_risk_pct = model_df["high_risk"].mean() * 100

def hist_by_outcome(ax, var, title, xlabel, bins=20):
    for val, col, lbl in [(0, "steelblue", "Low risk"), (1, "coral", "High risk")]:
        ax.hist(model_df[model_df["high_risk"] == val][var],
                bins=bins, alpha=0.6, color=col, label=lbl, density=True)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("Density", fontsize=9)
    ax.legend(fontsize=8)

def prop_bar(ax, categories, values, title, xlabel, bar_color="C0"):
    ax.cla()
    for i, (cat, val) in enumerate(zip(categories, values)):
        col = bar_color[i] if isinstance(bar_color, list) else bar_color
        ax.bar(cat, val, color=col, edgecolor="white", linewidth=0.5, width=0.55)
        ax.text(i, val + 0.4, f"{val:.1f}%", ha="center", fontsize=8)
    ax.axhline(overall_risk_pct, color="C3", linewidth=1.2,
               linestyle="--", label=f"Overall ({overall_risk_pct:.1f}%)")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(xlabel, fontsize=9)
    ax.set_ylabel("High risk (%)", fontsize=9)
    ax.legend(fontsize=8)

ax00 = fig2.add_subplot(gs2[0, 0])
risk_vig = model_df.groupby("vig_active")["high_risk"].mean() * 100
prop_bar(ax00, ["Inactive\n(0 days)", "Active\n(>=1 day)"],
         risk_vig.values, "High-risk % by vigorous activity",
         "Vigorous activity", bar_color=["lightgray", "C0"])

ax01 = fig2.add_subplot(gs2[0, 1])
hist_by_outcome(ax01, "mod_days", "Moderate activity days by outcome",
                "Days per week", bins=np.arange(-0.5, 8.5, 1))
ax01.set_xticks(range(0, 8))

ax02 = fig2.add_subplot(gs2[0, 2])
hist_by_outcome(ax02, "walk_days", "Walking days by outcome",
                "Days per week", bins=np.arange(-0.5, 8.5, 1))
ax02.set_xticks(range(0, 8))

ax10 = fig2.add_subplot(gs2[1, 0])
hist_by_outcome(ax10, "age", "Age by outcome", "Age (years)", bins=20)

ax11 = fig2.add_subplot(gs2[1, 1])
hist_by_outcome(ax11, "sedentary_hours", "Sedentary hours/day by outcome",
                "Hours per day", bins=20)

ax12 = fig2.add_subplot(gs2[1, 2])
risk_sleep = (model_df.groupby("sleep_label")["high_risk"]
                       .mean().reindex(sleep_order) * 100)
prop_bar(ax12, sleep_order, risk_sleep.values,
         "High-risk % by sleep duration", "Sleep duration")

tight_layout()

## 5. Frequentist Logistic Regression

As a reference model, we fit a standard logistic regression. Predictors include centred age, sex, centred sedentary hours, a binary vigorous activity indicator, moderate walking days, and sleep-duration dummies. Coefficients are reported alongside 95% confidence intervals and odds ratios.

In [ ]:
y = model_df["high_risk"]

def fit_logit_and_report(X, y):
    X_const = sm.add_constant(X)
    result  = sm.Logit(y, X_const).fit(disp=False)
    print(f"\n=== LOGISTIC REGRESSION RESULTS ===")
    print(result.summary())
    conf = result.conf_int()
    conf.columns = ["2.5%", "97.5%"]
    or_table = pd.DataFrame({
        "variable": result.params.index,
        "coef": result.params.values,
        "odds_ratio": np.exp(result.params.values),
        "ci_2.5": np.exp(conf["2.5%"].values),
        "ci_97.5": np.exp(conf["97.5%"].values),
        "p_value": result.pvalues.values,
    })
    return result, or_table

In [ ]:
lr = pd.concat([
    model_df[["age_c", "female", "sedentary_hours_c", "vig_active", "mod_days", "walk_days"]], sleep_dummies], axis=1)
result, or_table = fit_logit_and_report(lr, y)

## 6. Bayesian Logistic Regression

### 6.1 Prior Specification & Sensitivity Checks

We use weakly informative Normal(0, 0.5) priors on all slope coefficients and a Normal(0, 1.5) prior on the intercept. The plots below simulate prior predictive trajectories for three key predictors, age, sedentary hours, and walking days, to confirm that the priors allow a wide range of plausible probability curves without forcing extreme predictions.

$$
Y_i \sim \mathrm{Bernoulli}(p_i)
$$

$$
\mathrm{logit}(p_i)
=
\beta_0
+
\beta_{\mathrm{age}}\,\mathrm{age}_i
+
\beta_{\mathrm{sex}}\,\mathrm{sex}_i
+
\beta_{\mathrm{sed}}\,\mathrm{sedentary}_i
+
\beta_{\mathrm{vig}}\,\mathrm{vigorous}_i
+
\beta_{\mathrm{walk}}\,\mathrm{walking}_i
+
\text{sleep terms}
$$

$$
\beta_0 \sim \mathrm{Normal}(0,1.5),
\qquad
\beta_j \sim \mathrm{Normal}(0,0.5)
$$

In [ ]:
# Prior check
N_lines   = 50
age_range = linspace(-3, 3, 50)

beta_0_s   = normal(0, 1.5, N_lines)
beta_age_s = normal(0, 0.5, N_lines)

p_age = 1 / (1 + exp(-(beta_0_s[None, :] + beta_age_s[None, :] * age_range[:, None])))
plot(age_range, p_age, '0.7', alpha=0.5)
xlabel('centered age', fontsize=13)
ylabel('P(high risk)', fontsize=13)

In [ ]:
sed_range  = linspace(-3, 3, 50)
beta_sed_s = normal(0, 0.5, N_lines)

p_sed = 1 / (1 + exp(-(beta_0_s[None, :] + beta_sed_s[None, :] * sed_range[:, None])))
plot(sed_range, p_sed, '0.7', alpha=0.5)
xlabel('centered sedentary hours', fontsize=13)
ylabel('P(high risk)', fontsize=13)

In [ ]:
walk_range  = linspace(0, 7, 50)
beta_walk_s = normal(0, 0.5, N_lines)

p_walk = 1 / (1 + exp(-(beta_0_s[None, :] + beta_walk_s[None, :] * walk_range[:, None])))
plot(walk_range, p_walk, '0.7', alpha=0.5)
xlabel('walk days per week', fontsize=13)
ylabel('P(high risk)', fontsize=13)

### 6.2 Model Fitting

We fit the Bayesian logistic regression in PyMC using NUTS sampling. The model specification mirrors the frequentist model, with the priors defined above.

In [ ]:
with pm.Model() as diabetes_model:
    beta_0 = pm.Normal('beta_0', mu=0, sigma=1.5)
    beta_age = pm.Normal('beta_age', mu=0, sigma=0.5)
    beta_fem = pm.Normal('beta_fem', mu=0, sigma=0.5)
    beta_sed = pm.Normal('beta_sed', mu=0, sigma=0.5)
    beta_vig = pm.Normal('beta_vig', mu=0, sigma=0.5)
    beta_walk = pm.Normal('beta_walk', mu=0, sigma=0.5)
    beta_s1 = pm.Normal('beta_s1', mu=0, sigma=0.5)
    beta_s2 = pm.Normal('beta_s2', mu=0, sigma=0.5)
    beta_s3 = pm.Normal('beta_s3', mu=0, sigma=0.5)
    beta_s4 = pm.Normal('beta_s4', mu=0, sigma=0.5)
    _logit_p = (beta_0 + beta_age * model_df["age_c"] + beta_fem * model_df["female"] + beta_sed * model_df["sedentary_hours_c"]
                + beta_vig * model_df["vig_active"] + beta_walk * model_df["walk_days"] + beta_s1 * sleep_dummies["sleep_<=5h"]
                + beta_s2 * sleep_dummies["sleep_6h"] + beta_s3 * sleep_dummies["sleep_8h"] + beta_s4 * sleep_dummies["sleep_>=9h"])
    logit_p = pm.Deterministic('logit_p', _logit_p)
    Y_obs = pm.Bernoulli('Y_obs', logit_p=logit_p, observed=model_df["high_risk"])
    _diabetes_posterior = pm.sample(2000, tune=1000, chains=4)
diabetes_posterior = _diabetes_posterior.posterior.to_dataframe()
diabetes_posterior

### 6.3 Prior Predictive Check

We draw 500 samples from the prior predictive distribution to verify that the priors generate a reasonable spread of simulated high-risk prevalence values before seeing the data.

In [ ]:
with diabetes_model:
    prior_pred = pm.sample_prior_predictive(samples=500)

y_prior = prior_pred.prior_predictive["Y_obs"]
y_prior_mean = y_prior.values.mean(axis=2).flatten()

plt.hist(y_prior_mean, bins=30)
plt.xlabel("Simulated proportion of high risk")
plt.ylabel("Frequency")
plt.title("Prior predictive distribution of high-risk prevalence")
plt.show()

### 6.4 Posterior Inference

We summarise the posterior using a forest plot, a numerical summary table, and trace plots to assess MCMC convergence.

In [ ]:
az.plot_forest(_diabetes_posterior, var_names=['beta_age', 'beta_sed', 'beta_walk', 'beta_vig', 'beta_fem', 'beta_s1', 'beta_s2', 'beta_s3', 'beta_s4'], combined=True, figsize=[6, 5])
title('95% HDI', fontsize=12)

In [ ]:
print(az.summary(_diabetes_posterior, var_names=["~logit_p"], hdi_prob=0.95).to_string())

In [ ]:
az.plot_trace(_diabetes_posterior, var_names=['beta_age', 'beta_sed', 'beta_walk'], compact=True)
plt.tight_layout(h_pad=2)

In [ ]:
fig = figure(1, [15, 4])

for i, (param, label) in enumerate([('beta_age',  'Age'),
                                     ('beta_sed',  'Sedentary hours'),
                                     ('beta_walk', 'Walk days')]):
    fig.add_subplot(1, 3, i+1)
    sns.kdeplot(diabetes_posterior[param], fill=True, bw_adjust=3)
    axvline(diabetes_posterior[param].mean(), color='k',
            linewidth=1.5, linestyle='-', label='mean')
    xlabel('log-odds', fontsize=12)
    ylabel('density', fontsize=12)
    title(f'Posterior density: {label}', fontsize=12)
    legend(fontsize=9)

tight_layout()

### 6.5 Posterior Predictive Check

We compare the observed outcome distribution against replicated datasets drawn from the posterior predictive distribution to assess overall model fit.

In [ ]:
with diabetes_model:
    ppc = pm.sample_posterior_predictive(_diabetes_posterior)
az.plot_ppc(ppc)

## 7. Frequentist vs. Bayesian Comparison

The table below places frequentist coefficients and 95% confidence intervals alongside posterior means and 95% HDI from the Bayesian model. Both approaches use the same predictors, differences between estimates reflect the influence of the regularising priors in the Bayesian model.

In [ ]:
# Bayesian summary
bayes_summary = az.summary(_diabetes_posterior, var_names=["~logit_p"], hdi_prob=0.95)

# Map: frequentist variable name -> bayesian parameter name
name_map = {
    'const': 'beta_0',
    'age_c': 'beta_age',
    'female': 'beta_fem',
    'sedentary_hours_c': 'beta_sed',
    'vig_active': 'beta_vig',
    'walk_days': 'beta_walk',
    'sleep_<=5h': 'beta_s1',
    'sleep_6h': 'beta_s2',
    'sleep_8h': 'beta_s3',
    'sleep_>=9h': 'beta_s4',
}

rows = []
for var in result.params.index:
    row = {
        'Variable': var,
        'Freq_coef': f"{result.params[var]:.4f}",
        'Freq_95CI': f"[{result.conf_int().loc[var, 0]:.3f}, {result.conf_int().loc[var, 1]:.3f}]",
        'Freq_p': f"{result.pvalues[var]:.4f}" if result.pvalues[var] >= 0.001 else "<0.001",
    }
    bayes_name = name_map.get(var)
    if bayes_name and bayes_name in bayes_summary.index:
        b = bayes_summary.loc[bayes_name]
        row['Bayes_mean'] = f"{b['mean']:.4f}"
        row['Bayes_95HDI'] = f"[{b['hdi_2.5%']:.3f}, {b['hdi_97.5%']:.3f}]"
    else:
        row['Bayes_mean'] = '—'
        row['Bayes_95HDI'] = '—'
    rows.append(row)

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

### 7.1 Posterior Odds Ratios

We exponentiate the posterior samples to obtain odds ratios with 95% HDI for the key predictors.

In [ ]:
or_vars = {
    'beta_age':  'Age (per 1-year increase)',
    'beta_sed':  'Sedentary hours (per 1-hour increase)',
    'beta_walk': 'Walk days (per 1-day increase)',
    'beta_vig':  'Vigorous activity (active vs not)',
    'beta_fem':  'Female vs Male',
}

for var, label in or_vars.items():
    or_samples = np.exp(diabetes_posterior[var])
    print(f"{label}:")
    print(f"  OR = {np.median(or_samples):.3f}  "
          f"95% HDI [{np.percentile(or_samples,2.5):.3f}, {np.percentile(or_samples,97.5):.3f}]")

### 7.2 Scenario-Based Odds Ratios

We compute cumulative odds ratios for practically meaningful contrasts: walking every day of the week versus none, a 4-hour increase in daily sedentary time, and a 10-year increase in age.

In [ ]:
# Scenario Comparisons
walk_samples = diabetes_posterior['beta_walk']
or_walk_7 = np.exp(walk_samples * 7)
print(f"Walk 7 days/week vs 0:")
print(f"  OR = {np.median(or_walk_7):.3f}  "
      f"95% HDI [{np.percentile(or_walk_7,2.5):.3f}, {np.percentile(or_walk_7,97.5):.3f}]")

sed_samples = diabetes_posterior['beta_sed']
or_sed_4 = np.exp(sed_samples * 4)
print(f"\nSedentary +4 hours vs baseline:")
print(f"  OR = {np.median(or_sed_4):.3f}  "
      f"95% HDI [{np.percentile(or_sed_4,2.5):.3f}, {np.percentile(or_sed_4,97.5):.3f}]")

age_samples = diabetes_posterior['beta_age']
or_age_10 = np.exp(age_samples * 10)
print(f"\nAge +10 years:")
print(f"  OR = {np.median(or_age_10):.3f}  "
      f"95% HDI [{np.percentile(or_age_10,2.5):.3f}, {np.percentile(or_age_10,97.5):.3f}]")

## 8. Counterfactual Intervention Simulations

We use the posterior distribution to simulate two population-level interventions: (1) everyone walks 7 days per week, and (2) everyone reduces sedentary time by 2 hours per day. For each intervention, we compute the posterior distribution of the population-average prevalence of high diabetes risk and the absolute risk reduction relative to the factual scenario.

### 8.1 Single-Variable Interventions

We extract a subsample of 500 posterior draws and use the `compute_prevalence` function to estimate the posterior distribution of population-average high-risk prevalence under factual and counterfactual scenarios.

In [ ]:
# Extract and subsample posterior samples
np.random.seed(42)
idx = np.random.choice(len(diabetes_posterior['beta_0']), size=500, replace=False)

post = {}
for name in ['beta_0','beta_age','beta_fem','beta_sed','beta_vig','beta_walk',
             'beta_s1','beta_s2','beta_s3','beta_s4']:
    post[name] = diabetes_posterior[name].values[idx]

obs = {
    'age':  model_df["age_c"].values,
    'fem':  model_df["female"].values,
    'sed':  model_df["sedentary_hours_c"].values,
    'vig':  model_df["vig_active"].values,
    'walk': model_df["walk_days"].values,
    's1':   sleep_dummies["sleep_<=5h"].values,
    's2':   sleep_dummies["sleep_6h"].values,
    's3':   sleep_dummies["sleep_8h"].values,
    's4':   sleep_dummies["sleep_>=9h"].values,
}

In [ ]:
def compute_prevalence(post, obs_override=None):
    o = {**obs, **(obs_override or {})}
    logit_p = (post['beta_0'][:, None]
               + post['beta_age'][:, None]  * o['age'][None, :]
               + post['beta_fem'][:, None]  * o['fem'][None, :]
               + post['beta_sed'][:, None]  * o['sed'][None, :]
               + post['beta_vig'][:, None]  * o['vig'][None, :]
               + post['beta_walk'][:, None] * o['walk'][None, :]
               + post['beta_s1'][:, None]   * o['s1'][None, :]
               + post['beta_s2'][:, None]   * o['s2'][None, :]
               + post['beta_s3'][:, None]   * o['s3'][None, :]
               + post['beta_s4'][:, None]   * o['s4'][None, :])
    p = 1 / (1 + np.exp(-logit_p))
    return p.mean(axis=1)

In [ ]:
# Factual and counterfactual simulations
prev_factual    = compute_prevalence(post)
prev_walk7      = compute_prevalence(post, {'walk': np.full_like(obs['walk'], 7.0)})
prev_sed_minus2 = compute_prevalence(post, {'sed':  obs['sed'] - 2.0})

In [ ]:
# Results
def report(label, prev):
    med = np.median(prev) * 100
    lo  = np.percentile(prev, 2.5) * 100
    hi  = np.percentile(prev, 97.5) * 100
    print(f"{label}:")
    print(f"  Prevalence = {med:.1f}%  95% CI [{lo:.1f}%, {hi:.1f}%]")

report("Factual (observed)", prev_factual)
report("Intervention 1: everyone walks 7 days/week", prev_walk7)
report("Intervention 2: everyone reduces sedentary time by 2h", prev_sed_minus2)

arr_walk = prev_factual - prev_walk7
arr_sed  = prev_factual - prev_sed_minus2

print(f"\nARR (walk 7 days): {np.median(arr_walk)*100:.1f} pp  "
      f"95% CI [{np.percentile(arr_walk,2.5)*100:.1f}, {np.percentile(arr_walk,97.5)*100:.1f}]")
print(f"ARR (sedentary -2h): {np.median(arr_sed)*100:.1f} pp  "
      f"95% CI [{np.percentile(arr_sed,2.5)*100:.1f}, {np.percentile(arr_sed,97.5)*100:.1f}]")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].hist(prev_factual * 100, bins=40, alpha=0.5, label='Factual', color='gray')
axes[0].hist(prev_walk7 * 100, bins=40, alpha=0.5, label='Walk 7 days/week', color='steelblue')
axes[0].set_xlabel('Population high-risk prevalence (%)')
axes[0].set_ylabel('Posterior draws')
axes[0].set_title('Intervention 1: Universal walking')
axes[0].legend()

axes[1].hist(prev_factual * 100, bins=40, alpha=0.5, label='Factual', color='gray')
axes[1].hist(prev_sed_minus2 * 100, bins=40, alpha=0.5, label='Sedentary -2h', color='coral')
axes[1].set_xlabel('Population high-risk prevalence (%)')
axes[1].set_ylabel('Posterior draws')
axes[1].set_title('Intervention 2: Reduce sedentary time')
axes[1].legend()

plt.tight_layout()
plt.show()

### 8.2 Combined Intervention

This section simulates a combined lifestyle intervention in which every individual walks at least 4 days per week and limits sedentary time to no more than 6 hours per day. We compare the resulting posterior distribution of population risk against the observed reality.

In [ ]:
cf_df = model_df.copy()
# intervention: walk at least 4 days a week, sit for no greater than 6 hours per day
cf_df["walk_days"] = np.maximum(cf_df["walk_days"], 4)
cf_df["sedentary_hours_raw"] = np.minimum(model_df["sedentary_hours"], 6)
cf_df["sedentary_hours_c"] = cf_df["sedentary_hours_raw"] - model_df["sedentary_hours"].mean()

# randomly select 1000 draws from the posterior, set seed for replicability
n_samples = 1000
np.random.seed(2)
sample_idx = np.random.choice(len(diabetes_posterior), size=n_samples, replace=False)
post_samples = diabetes_posterior.iloc[sample_idx]

# calculate counterfactual log-odds for every person, for every posterior draw (shape: 1501 people, 1000 posterior draws)
cf_logit_matrix = np.zeros((len(cf_df), n_samples))
for i, (_, row) in enumerate(post_samples.iterrows()):
    cf_logit = (
        row['beta_0']
        + row['beta_age'] * cf_df["age_c"]
        + row['beta_fem'] * cf_df["female"]
        + row['beta_sed'] * cf_df["sedentary_hours_c"] # intervened with cap
        + row['beta_vig'] * cf_df["vig_active"]
        + row['beta_walk'] * cf_df["walk_days"] # intervened with lower bound
        + row['beta_s1'] * sleep_dummies["sleep_<=5h"]
        + row['beta_s2'] * sleep_dummies["sleep_6h"]
        + row['beta_s3'] * sleep_dummies["sleep_8h"]
        + row['beta_s4'] * sleep_dummies["sleep_>=9h"]
    )
    cf_logit_matrix[:, i] = cf_logit

# convert log-odds to probabilities using the inverse logit (expit)
cf_prob_matrix = expit(cf_logit_matrix)

# calculate the population average risk for both scenarios
real_risk_mean = model_df["high_risk"].mean()
cf_pop_risk = cf_prob_matrix.mean(axis=0)



In [ ]:
# plot the result
plt.figure(figsize=(8, 5))
sns.kdeplot(cf_pop_risk, fill=True, color="C1", label="Counterfactual (Walk >= 4 days, Sedentary <= 6 hours)")
plt.axvline(real_risk_mean, color="k", linestyle="--", label=f"Observed Reality ({real_risk_mean:.1%})")

plt.title("Counterfactual Forecasting: Population Diabetes Risk")
plt.xlabel("Population High-Risk Probability")
plt.ylabel("Density")
plt.legend()
plt.gca().xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.show()